In [ ]:
# 1) Gerekli kutuphaneler
!pip -q install openpyxl scipy numpy numba
print("Kurulum tamamlandi.")

# v6.2: product transition/start Prep_min validation enabled.


In [ ]:
# 2) GIRIS EXCEL KAYNAGI: PATH veya GITHUB REPOSITORY
#
# En kolay kullanim: sadece GitHub repository adresini yazin.
# Kod repository'nin main branch'indeki
# production_planning_input_v6_excel_only.xlsx dosyasini otomatik indirir.
#
# Ornek:
# INPUT_SOURCE = "https://github.com/hadokan/Production-planning"
#
# Yerel / Colab path de kullanabilirsiniz:
# INPUT_SOURCE = "/content/production_planning_input_v6_excel_only.xlsx"
#
# Google Drive path de kullanabilirsiniz:
# INPUT_SOURCE = "/content/drive/MyDrive/Planlama/production_planning_input_v6_excel_only.xlsx"

from pathlib import Path
from urllib.parse import urlparse
from urllib.request import Request, urlopen
import shutil

INPUT_SOURCE = "https://github.com/hadokan/Production-planning"
GITHUB_INPUT_FILENAME = "production_planning_input_v6_excel_only.xlsx"
DEFAULT_GITHUB_BRANCH = "main"

def github_source_to_raw(url: str, filename: str = GITHUB_INPUT_FILENAME) -> tuple[str, str]:
    """
    Desteklenen GitHub adresleri:
      1) https://github.com/owner/repo
      2) https://github.com/owner/repo/tree/branch
      3) https://github.com/owner/repo/blob/branch/file.xlsx
      4) raw.githubusercontent.com adresi

    Repository veya tree linkinde filename otomatik eklenir.
    Donus: (raw_url, local_filename)
    """
    parsed = urlparse(url)
    host = parsed.netloc.lower()
    parts = [p for p in parsed.path.split("/") if p]

    if host == "raw.githubusercontent.com":
        local_name = Path(parsed.path).name
        if not local_name.lower().endswith(".xlsx"):
            raise ValueError("RAW GitHub linki bir .xlsx dosyasini gostermeli.")
        return url, local_name

    if host != "github.com":
        raise ValueError("URL girdisi GitHub repository/dosya linki olmali veya yerel path kullanilmalidir.")

    if len(parts) < 2:
        raise ValueError("GitHub repository adresi gecersiz.")

    owner, repo = parts[0], parts[1]

    # Dosya linki: /owner/repo/blob/branch/path/to/file.xlsx
    if len(parts) >= 5 and parts[2] == "blob":
        branch = parts[3]
        file_path = "/".join(parts[4:])
        local_name = Path(file_path).name
        if not local_name.lower().endswith(".xlsx"):
            raise ValueError("GitHub blob linki bir .xlsx dosyasini gostermeli.")
        raw = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{file_path}"
        return raw, local_name

    # Branch/tree linki: /owner/repo/tree/branch[/optional/folder]
    if len(parts) >= 4 and parts[2] == "tree":
        branch = parts[3]
        folder = "/".join(parts[4:])
        file_path = f"{folder}/{filename}" if folder else filename
        raw = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{file_path}"
        return raw, filename

    # Repository ana sayfasi: /owner/repo
    branch = DEFAULT_GITHUB_BRANCH
    raw = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{filename}"
    return raw, filename

def resolve_input(source: str) -> Path:
    source = str(source).strip()
    if not source:
        raise ValueError("INPUT_SOURCE bos olamaz.")

    # GitHub URL ise repository veya dosya linkini cozumle ve Excel'i indir.
    if source.startswith(("http://", "https://")):
        download_url, filename = github_source_to_raw(source)
        target = Path("/content") / filename
        print("GitHub Excel indiriliyor:", download_url)
        req = Request(download_url, headers={"User-Agent": "Mozilla/5.0"})
        try:
            with urlopen(req, timeout=60) as response, open(target, "wb") as f:
                shutil.copyfileobj(response, f)
        except Exception as exc:
            raise RuntimeError(
                "GitHub Excel dosyasi indirilemedi. Repository public olmali ve "
                f"'{filename}' dosyasi secilen branch'te bulunmali. URL: {download_url}"
            ) from exc
        if not target.exists() or target.stat().st_size == 0:
            raise RuntimeError("Indirilen Excel dosyasi bos.")
        print("Giris Excel hazir:", target)
        return target

    # Path ise Google Drive gerekiyorsa mount et.
    path = Path(source).expanduser()
    if str(path).startswith("/content/drive/"):
        from google.colab import drive
        drive.mount("/content/drive")

    if not path.exists():
        raise FileNotFoundError(f"Giris Excel bulunamadi: {path}")
    if path.suffix.lower() != ".xlsx":
        raise ValueError(f"Giris dosyasi .xlsx olmali: {path}")
    return path

INPUT_XLSX = resolve_input(INPUT_SOURCE)
OUTPUT_XLSX = INPUT_XLSX.with_name(INPUT_XLSX.stem + "_output.xlsx")
print("INPUT_XLSX :", INPUT_XLSX)
print("OUTPUT_XLSX:", OUTPUT_XLSX)


In [ ]:
# 3) TUM PLANLAMA / OPTIMIZASYON KODU
# Ayrica .py, JSON veya config dosyasi gerekmez.
#!/usr/bin/env python3
"""Excel v6 -> constructive search + unrestricted MILP -> five-sheet Excel.

No ChatGPT/API connection or commercial solver is required. All computation is
local. Read README_TR.md for supported schema, numerical conventions and limits.
"""
from __future__ import annotations

import argparse
import hashlib
import logging
import math
import os
import platform
import re
import sys
import time
import warnings
import zipfile
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
from xml.etree import ElementTree as ET

import numpy as np
import scipy
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.comments import Comment
from openpyxl.utils import get_column_letter
from openpyxl.workbook.properties import CalcProperties

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        def wrapper(f):
            return f
        return wrapper

VERSION = "6.2.0"
TOL = 1e-5                 # Validation tolerance in tons / hours.
POSITIVE_T = 1e-4          # Numerical activation floor: 0.1 kg, NOT a batch size.
LOG = logging.getLogger("production_planner")

SCHEMA = {
    "products": "Product_ID Product Family Salt_Status Sequence_Order Priority".split(),
    "machines": "Machine_ID Machine Active Priority".split(),
    "machine_product": "Machine_ID Product_ID Allowed MachinePriority Capacity_tph MinRun_t MaxDaily_t Preparation_min".split(),
    "targets": "Product_ID Product Target_t TargetPriority TargetMode".split(),
    "calendar": "Machine_ID Day Shift Available Hours Reason".split(),
    "sequence_rules": "Rule_ID Family Before_Product_ID After_Product_ID Scope Hard".split(),
    "changeover_rules": "Rule_ID Scope Max_Products_Per_Shift Max_Changeovers_Per_Day Hard Description".split(),
    "allocation_rules": "Rule_ID Scope Prefer_Single_Machine Preferred_Max_Machines Allow_Split_If_Required Hard Description Objective_ID".split(),
    "initial_state": "Machine_ID Current_Product_ID Run_Produced_t Setup_Completed".split(),
    "objectives": "Priority Objective_ID Direction Enabled Description".split(),
    "settings": "Parameter Value Description".split(),
}
SETTING_NAMES = set("""SchemaVersion ShiftHours DefaultPreparationMin MaxPlanningDays
AllowFinalRunBelowMinimum AllowSurplusProduction PreventProductReturn
PreferContinuousProduction CriticalProductFirst ObjectiveMode UnmetTolerance_t
UnmetToleranceScope UnmetReferenceMode CountCrossDayChangeovers
ChangeoverAttribution CountInitialSetupAsChangeover IdleResetsProductState
SequenceRuleSource SequenceConflictPolicy OptimizationMode HeuristicTrials
MILPPolishTimeSec MILPNeighborhoodMode RuntimeLogging InputValidation
UnknownParameterPolicy RandomSeed""".split())
OBJECTIVE_NAMES = ["MIN_UNMET", "MIN_PRODUCT_MACHINE_SPLITS", "MIN_CHANGEOVERS",
    "MIN_FAMILY_MACHINE_SPLITS", "MIN_PRODUCTS_PER_MACHINE", "MIN_SURPLUS",
    "MIN_MACHINE_PRIORITY_COST"]

class InputError(ValueError):
    """Input is missing, inconsistent, or outside the implemented v6 contract."""


def text(v: Any) -> str:
    return " ".join(str(v).strip().split()) if v is not None else ""


def number(v: Any, name: str, minimum: float = 0, integer: bool = False) -> float:
    if isinstance(v, bool):
        raise InputError(f"{name}: use numeric 0/1, not a Boolean cell.")
    try:
        f = float(v)
    except (ValueError, TypeError) as exc:
        raise InputError(f"{name}: expected a number; got {v!r}") from exc
    if not math.isfinite(f) or f < minimum or (integer and f != int(f)):
        raise InputError(f"{name}: invalid numeric value {v!r}")
    return int(f) if integer else f


def flag(v: Any, name: str) -> int:
    k = number(v, name, integer=True)
    if k not in (0, 1):
        raise InputError(f"{name}: expected 0 or 1.")
    return k


def read_tables(path: Path) -> dict[str, list[dict]]:
    wb = load_workbook(path, data_only=False, read_only=True)
    try:
        if set(wb.sheetnames) != set(SCHEMA):
            raise InputError(f"Sheet mismatch. Missing={set(SCHEMA)-set(wb.sheetnames)}; "
                             f"unknown={set(wb.sheetnames)-set(SCHEMA)}")
        tables = {}
        for name, expected in SCHEMA.items():
            rows = list(wb[name].iter_rows(values_only=True))
            if not rows:
                raise InputError(f"{name}: empty sheet.")
            raw = list(rows[0])
            while raw and raw[-1] is None:
                raw.pop()
            headers = [text(v) for v in raw]
            if len(headers) != len(set(headers)) or set(headers) != set(expected):
                raise InputError(f"{name}: expected headers {expected}; got {headers}")
            data = []
            for rn, vals in enumerate(rows[1:], 2):
                if not any(v is not None for v in vals):
                    continue
                if any(v is not None for v in vals[len(headers):]):
                    raise InputError(f"{name}!row {rn}: data beyond known columns.")
                d = dict(zip(headers, vals))
                if any(isinstance(v, str) and v.startswith("=") for v in d.values()):
                    raise InputError(f"{name}!row {rn}: input cells must be values, not formulas.")
                d["_row"] = rn
                data.append(d)
            tables[name] = data
        return tables
    finally:
        wb.close()


def unique(rows: list[dict], keys: tuple[str, ...], sheet: str) -> None:
    seen = set()
    for r in rows:
        key = tuple(text(r[k]) for k in keys)
        if any(not x for x in key) or key in seen:
            raise InputError(f"{sheet}: empty or duplicate key {key}")
        seen.add(key)


@dataclass
class Data:
    path: Path
    sha256: str
    tables: dict
    settings: dict
    products: list[dict]
    machines: list[dict]
    slots: list[tuple[int, int]]
    families: list[str]
    family_index: np.ndarray
    rate: np.ndarray
    minimum: np.ndarray
    daily: np.ndarray
    prep: np.ndarray
    priority: np.ndarray
    hours: np.ndarray
    target: np.ndarray
    hard: np.ndarray
    predecessors: np.ndarray
    initial: np.ndarray
    initial_qty: np.ndarray
    prepared: np.ndarray
    max_changes: np.ndarray
    product_limits: np.ndarray
    family_limits: np.ndarray
    objectives: list[str]
    warnings: list[str] = field(default_factory=list)

    @property
    def M(self): return len(self.machines)
    @property
    def P(self): return len(self.products)
    @property
    def T(self): return len(self.slots)


def load_input(path: str | Path) -> Data:
    path = Path(path).expanduser().resolve()
    if not path.is_file() or path.suffix.lower() != ".xlsx":
        raise InputError(f"Input .xlsx not found: {path}")
    tb = read_tables(path)
    for sh, keys in {"products": ("Product_ID",), "machines": ("Machine_ID",),
         "machine_product": ("Machine_ID", "Product_ID"), "targets": ("Product_ID",),
         "calendar": ("Machine_ID", "Day", "Shift"), "sequence_rules": ("Rule_ID",),
         "changeover_rules": ("Rule_ID",), "allocation_rules": ("Rule_ID",),
         "initial_state": ("Machine_ID",), "objectives": ("Priority",),
         "settings": ("Parameter",)}.items():
        unique(tb[sh], keys, sh)
    s = {text(r['Parameter']): r['Value'] for r in tb['settings']}
    if set(s) != SETTING_NAMES:
        raise InputError(f"Settings mismatch. Missing={SETTING_NAMES-set(s)}; unknown={set(s)-SETTING_NAMES}")
    required_values = {
        "SchemaVersion": "6.0", "ObjectiveMode": "LEXICOGRAPHIC_WITH_UNMET_TOLERANCE",
        "UnmetToleranceScope": "TOTAL_PLAN", "UnmetReferenceMode": "BEST_FOUND_BASELINE",
        "ChangeoverAttribution": "NEW_PRODUCT_START_DAY", "SequenceRuleSource": "sequence_rules",
        "SequenceConflictPolicy": "ERROR", "OptimizationMode": "HYBRID",
        "MILPNeighborhoodMode": "REALLOCATION_AND_RESEQUENCING", "InputValidation": "STRICT",
        "UnknownParameterPolicy": "ERROR"}
    for key, wanted in required_values.items():
        if text(s[key]) != wanted:
            raise InputError(f"Unsupported {key}={s[key]!r}. This release supports {wanted!r}.")
    for key in ['AllowFinalRunBelowMinimum','PreferContinuousProduction','CriticalProductFirst','RuntimeLogging']:
        s[key] = flag(s[key], key)
    for key, wanted in {"AllowSurplusProduction":0,"PreventProductReturn":1,
             "CountCrossDayChangeovers":1,"CountInitialSetupAsChangeover":0,"IdleResetsProductState":0}.items():
        if flag(s[key], key) != wanted:
            raise InputError(f"Unsupported {key}={s[key]}; this release requires {wanted}.")
    for key in ['DefaultPreparationMin','MILPPolishTimeSec','UnmetTolerance_t']:
        s[key] = number(s[key], key)
    s['ShiftHours'] = number(s['ShiftHours'], 'ShiftHours', minimum=1e-9)
    for key in ['MaxPlanningDays','HeuristicTrials']:
        s[key] = number(s[key], key, minimum=1, integer=True)
    s['RandomSeed'] = number(s['RandomSeed'], 'RandomSeed', minimum=0, integer=True)
    if s['RandomSeed'] > 2147483646:
        raise InputError('RandomSeed must be between 0 and 2147483646.')
    products = tb['products']
    if s['RuntimeLogging'] != 1:
        raise InputError('This audited release requires RuntimeLogging=1.')
    if not products or not tb['machines']:
        raise InputError('At least one product and one machine are required.')
    for r in products:
        for k in ['Product_ID','Product','Family','Salt_Status']:
            r[k] = text(r[k])
            if not r[k]: raise InputError(f"products: {k} is empty.")
        if r['Salt_Status'] not in ['UNSALTED','SALTED','NONE']:
            raise InputError(f"Invalid Salt_Status for {r['Product_ID']}")
        r['Sequence_Order'] = number(r['Sequence_Order'],'Sequence_Order',minimum=1,integer=True)
        r['Priority'] = number(r['Priority'],'Product.Priority',minimum=1,integer=True)
    if len({r['Priority'] for r in products}) > 1:
        raise InputError('Mixed product Priority tiers are not implemented; equal values are required in this v6 release.')
    allmachines = {text(r['Machine_ID']): r for r in tb['machines']}
    machines = []
    for r in tb['machines']:
        r['Machine_ID'],r['Machine'] = text(r['Machine_ID']),text(r['Machine'])
        r['Priority'] = number(r['Priority'],'Machine.Priority',minimum=1,integer=True)
        r['Active'] = flag(r['Active'],'Active')
        if r['Active']: machines.append(r)
    if not machines: raise InputError('No active machines.')
    pm = {r['Product_ID']: i for i,r in enumerate(products)}
    mm = {r['Machine_ID']: i for i,r in enumerate(machines)}
    M,P = len(machines),len(products)
    arrays = [np.zeros((M,P),dtype=float) for _ in range(5)]
    rate,minimum,daily,prep,priority = arrays
    for r in tb['machine_product']:
        mid,pid = text(r['Machine_ID']),text(r['Product_ID'])
        if mid not in allmachines or pid not in pm: raise InputError(f"Unknown machine-product: {mid}/{pid}")
        allowed = flag(r['Allowed'],'Allowed')
        vals = [number(r[k],f'{mid}/{pid}/{k}') for k in ['Capacity_tph','MinRun_t','MaxDaily_t']]
        pr = number(r['Preparation_min'] if r['Preparation_min'] is not None else s['DefaultPreparationMin'],'Preparation_min')
        mp = number(r['MachinePriority'],'MachinePriority',minimum=1,integer=True) if allowed else 0
        if allowed and (vals[0] <= 0 or vals[2] <= 0):
            raise InputError(f'{mid}/{pid}: allowed pair needs positive rate and MaxDaily_t.')
        if not allowed and any(vals): raise InputError(f'{mid}/{pid}: disallowed pair must have zero rate/run/daily quantities.')
        if mid in mm and allowed:
            m,p=mm[mid],pm[pid]
            rate[m,p],minimum[m,p],daily[m,p],prep[m,p],priority[m,p]=*vals,pr,mp
    target,hard=np.zeros(P),np.zeros(P,dtype=np.int64)
    if {text(r['Product_ID']) for r in tb['targets']} != set(pm):
        raise InputError('targets must contain exactly one row for each product (zero target is allowed).')
    target_priorities=[]
    for r in tb['targets']:
        p=pm[text(r['Product_ID'])]
        if text(r['Product']) != products[p]['Product']:
            raise InputError(f"targets.Product conflicts with products for {r['Product_ID']}")
        target[p]=number(r['Target_t'],'Target_t')
        if 0<target[p]<POSITIVE_T: raise InputError('Target below numerical activation floor of 0.0001 t.')
        mode=text(r['TargetMode'])
        if mode not in ['SOFT','HARD']: raise InputError('TargetMode must be SOFT or HARD.')
        hard[p]=(mode=='HARD')
        target_priorities.append(number(r['TargetPriority'],'TargetPriority',minimum=1,integer=True))
    if len(set(target_priorities))>1:
        raise InputError('This v6 release supports equal TargetPriority values. Mixed target tiers need an explicitly agreed tolerance policy; not silently weighted.')
    slots_set=set(); cal={}
    for r in tb['calendar']:
        mid=text(r['Machine_ID'])
        match=re.fullmatch(r'Day-(\d+)',text(r['Day']))
        if mid not in allmachines or not match: raise InputError('calendar: unknown machine or invalid Day-n label.')
        day=int(match.group(1)); shift=number(r['Shift'],'Shift',minimum=1,integer=True)
        if not 1<=day<=s['MaxPlanningDays']: raise InputError('Calendar exceeds MaxPlanningDays.')
        available=flag(r['Available'],'Available'); hours=number(r['Hours'],'Hours')
        if (not available and hours!=0) or (available and hours<=0): raise InputError('Calendar Available/Hours conflict.')
        if mid in mm:
            key=(mm[mid],day,shift)
            if key in cal: raise InputError('Duplicate normalized calendar key.')
            cal[key]=hours
            slots_set.add((day,shift))
    slots=sorted(slots_set); T=len(slots)
    if not T: raise InputError('Empty active calendar.')
    hours=np.zeros((M,T))
    for m in range(M):
        for t,(day,shift) in enumerate(slots):
            if (m,day,shift) not in cal: raise InputError(f'Missing calendar row for {machines[m]["Machine_ID"]}/Day-{day}/{shift}; add explicit closure.')
            hours[m,t]=cal[m,day,shift]
    pre=np.zeros((P,P),dtype=np.int64)
    for r in tb['sequence_rules']:
        a,b=text(r['Before_Product_ID']),text(r['After_Product_ID'])
        if a not in pm or b not in pm or a==b: raise InputError('Invalid sequence pair.')
        if text(r['Scope'])!='SAME_MACHINE' or flag(r['Hard'],'Sequence.Hard')!=1:
            raise InputError('Only HARD SAME_MACHINE sequence rules are supported.')
        pa,pb=products[pm[a]],products[pm[b]]
        if pa['Family']!=pb['Family'] or text(r['Family'])!=pa['Family'] or pa['Sequence_Order']>=pb['Sequence_Order']:
            raise InputError(f'Sequence rule conflicts with Family/Sequence_Order: {a}->{b}')
        if pre[pm[a],pm[b]]: raise InputError('Duplicate sequence pair.')
        pre[pm[a],pm[b]]=1
    for a in range(P):
        for b in range(P):
            if products[a]['Family']==products[b]['Family'] and products[a]['Sequence_Order']<products[b]['Sequence_Order'] and not pre[a,b]:
                raise InputError(f'Missing explicit sequence pair {products[a]["Product_ID"]}->{products[b]["Product_ID"]}')
    initial=np.full(M,-1,dtype=np.int64); initial_qty=np.zeros(M); prepared=np.zeros(M,dtype=np.int64)
    if {text(r['Machine_ID']) for r in tb['initial_state']}!=set(allmachines):
        raise InputError('initial_state needs one row for each machine.')
    for r in tb['initial_state']:
        mid=text(r['Machine_ID']); pid=text(r['Current_Product_ID'])
        if pid and pid not in pm: raise InputError('Unknown initial product.')
        q=number(r['Run_Produced_t'],'Run_Produced_t'); ready=flag(r['Setup_Completed'],'Setup_Completed')
        if not pid and (q or ready): raise InputError('Empty initial product requires zero Run_Produced_t and Setup_Completed.')
        if mid in mm:
            m=mm[mid]
            if pid:
                p=pm[pid]
                if rate[m,p]<=0: raise InputError('Initial product is not allowed on its machine.')
                # Abandoning an incomplete carried run requires a separate terminal-state contract.
                if q>0 and q+TOL<minimum[m,p]:
                    raise InputError(f'{mid}: incomplete carried run is not supported in this release. Do not drop its balance silently; model a locked continuation first.')
                initial[m]=p
            initial_qty[m]=q; prepared[m]=ready
    cr=tb['changeover_rules']
    if len(cr)!=1 or text(cr[0]['Scope'])!='MACHINE_DAY': raise InputError('Use one global MACHINE_DAY changeover rule.')
    if number(cr[0]['Max_Products_Per_Shift'],'Max_Products_Per_Shift',integer=True)!=1 or flag(cr[0]['Hard'],'Changeover.Hard')!=1:
        raise InputError('This release requires HARD one-product-per-machine-shift.')
    maxchg=number(cr[0]['Max_Changeovers_Per_Day'],'Max_Changeovers_Per_Day',integer=True)
    max_changes=np.full(M,maxchg,dtype=np.int64)
    families=list(dict.fromkeys(r['Family'] for r in products))
    family_index=np.array([families.index(r['Family']) for r in products],dtype=np.int64)
    product_limits=np.full(P,M,dtype=np.int64); family_limits=np.full(len(families),M,dtype=np.int64)
    for r in tb['allocation_rules']:
        scope=text(r['Scope']); oid=text(r['Objective_ID'])
        preferred=number(r['Preferred_Max_Machines'],'Preferred_Max_Machines',minimum=1,integer=True)
        prefer=flag(r['Prefer_Single_Machine'],'Prefer_Single_Machine')
        split=flag(r['Allow_Split_If_Required'],'Allow_Split_If_Required'); ishard=flag(r['Hard'],'Allocation.Hard')
        if not prefer or preferred!=1: raise InputError('Only a one-machine preference/limit is implemented; unsupported allocation values are rejected.')
        if scope=='ALL_PRODUCTS': ids=list(range(P)); dest=product_limits; expected=OBJECTIVE_NAMES[1]
        elif scope.startswith('PRODUCT:') and scope[8:] in pm: ids=[pm[scope[8:]]]; dest=product_limits; expected=OBJECTIVE_NAMES[1]
        elif scope=='ALL_FAMILIES': ids=list(range(len(families))); dest=family_limits; expected=OBJECTIVE_NAMES[3]
        elif scope.startswith('FAMILY:') and scope[7:] in families: ids=[families.index(scope[7:])]; dest=family_limits; expected=OBJECTIVE_NAMES[3]
        else: raise InputError(f'Unsupported allocation scope {scope}')
        if oid!=expected: raise InputError('Allocation Objective_ID mismatch.')
        if ishard or not split:
            for i in ids: dest[i]=min(dest[i],preferred)
    objs=[]; seen=set()
    for r in sorted(tb['objectives'],key=lambda z:number(z['Priority'],'Objective.Priority',minimum=1,integer=True)):
        oid=text(r['Objective_ID'])
        if oid not in OBJECTIVE_NAMES or oid in seen or text(r['Direction'])!='MIN': raise InputError('Unknown/duplicate objective or unsupported direction.')
        seen.add(oid)
        if flag(r['Enabled'],'Enabled'): objs.append(oid)
    if not objs or objs[0]!='MIN_UNMET': raise InputError('MIN_UNMET must be the first enabled objective.')
    notes=[]
    for p in range(P):
        if target[p]>0 and not np.any(rate[:,p]>0):
            notes.append(f"NO_ELIGIBLE_MACHINE: {products[p]['Product_ID']}")
    return Data(path,hashlib.sha256(path.read_bytes()).hexdigest(),tb,s,products,machines,slots,
        families,family_index,rate,minimum,daily,prep,priority,hours,target,hard,pre,
        initial,initial_qty,prepared,max_changes,product_limits,family_limits,objs,notes)


@dataclass
class Plan:
    product: np.ndarray
    quantity: np.ndarray
    method: str = ''
    metrics: dict = field(default_factory=dict)

    def clone(self):
        return Plan(self.product.copy(),self.quantity.copy(),self.method,dict(self.metrics))


def evaluate(d: Data, plan: Plan) -> dict:
    amounts=np.zeros(d.P); assigned=np.zeros((d.M,d.P),dtype=int)
    changes=0; run_count=0; pcost=0.; mastercost=0.; idle_gaps=0
    for m in range(d.M):
        last=int(d.initial[m]); seen=set(); had=False; idle_since=False
        for t in range(d.T):
            p=int(plan.product[m,t]); q=float(plan.quantity[m,t])
            if p<0 or q<=TOL:
                if had and d.hours[m,t]>0: idle_since=True
                continue
            if idle_since and p==last: idle_gaps+=1
            idle_since=False; had=True
            amounts[p]+=q; assigned[m,p]=1
            pcost+=q*(d.priority[m,p]-1)
            mastercost+=q*(d.machines[m]['Priority']-1)
            if p not in seen: run_count+=1; seen.add(p)
            if last>=0 and p!=last: changes+=1
            last=p
    uses=assigned.sum(axis=0); famuses=[]
    for f in range(len(d.families)):
        famuses.append(int(np.any(assigned[:,d.family_index==f],axis=1).sum()))
    result={
        'target':float(d.target.sum()),'planned':float(amounts.sum()),
        'unmet':float(np.maximum(d.target-amounts,0).sum()),
        'surplus':float(np.maximum(amounts-d.target,0).sum()),
        'product_splits':int(np.maximum(uses-1,0).sum()),
        'split_products':int((uses>1).sum()),'changeovers':changes,
        'family_splits':sum(max(x-1,0) for x in famuses),
        'assignments':int(assigned.sum()),'runs':run_count,
        'machines_used':int(np.any(assigned,axis=1).sum()),
        'priority_cost':float(pcost),'master_priority_cost':float(mastercost),'idle_gaps':idle_gaps,
    }
    plan.metrics=result
    return result


def preference_key(d: Data, met: dict) -> tuple:
    lookup={'MIN_UNMET':'unmet','MIN_PRODUCT_MACHINE_SPLITS':'product_splits',
        'MIN_CHANGEOVERS':'changeovers','MIN_FAMILY_MACHINE_SPLITS':'family_splits',
        'MIN_PRODUCTS_PER_MACHINE':'assignments','MIN_SURPLUS':'surplus',
        'MIN_MACHINE_PRIORITY_COST':'priority_cost'}
    k=tuple(round(met[lookup[o]],6) for o in d.objectives if o!='MIN_UNMET')
    return k+(round(met['unmet'],6),round(met['master_priority_cost'],6),
              met['idle_gaps'] if d.settings['PreferContinuousProduction'] else 0)


def expected_preparation_min(d: Data, m: int, p: int, last_p: int, first_prod: bool) -> float:
    """Expected preparation minutes for this production start.

    Rules:
    - Same product continuation: 0 min.
    - Empty/other-product -> new product: machine_product.Preparation_min.
    - If the machine is already prepared for the same initial product, first start may be 0.
    - IDLE/CLOSED slots never reset the last produced product.
    """
    if p < 0:
        return 0.0
    if (not first_prod) and p == last_p:
        return 0.0
    if first_prod and p == int(d.initial[m]) and bool(d.prepared[m]):
        return 0.0
    return float(d.prep[m, p])


def validate_detail_preparation(d: Data, rows: list[dict]) -> list[dict]:
    """Validate exported Prep_min against the production sequence.

    This is deliberately independent from the Excel formulas. A product transition
    on the same machine must show exactly the Preparation_min from the input Excel.
    """
    bad=[]
    by_machine=defaultdict(list)
    for r in rows:
        by_machine[int(r['m'])].append(r)
    for m, rr in by_machine.items():
        rr=sorted(rr,key=lambda z:int(z['t']))
        last=int(d.initial[m]); first_prod=True
        for r in rr:
            p=int(r['p'])
            if p < 0:
                continue
            expected=expected_preparation_min(d,m,p,last,first_prod)
            actual=float(r['Preparation_min'])
            if abs(actual-expected)>TOL:
                bad.append({
                    'code':'PREPARATION_MIN',
                    'message':f"{r['Machine']}/{r['Day']}/S{r['Shift']}: Prep_min={actual:.6f}, expected={expected:.6f}"
                })
            # Explicit transition guard requested by the user.
            if last >= 0 and p != last and abs(actual-float(d.prep[m,p]))>TOL:
                bad.append({
                    'code':'CHANGEOVER_PREP',
                    'message':f"{r['Machine']}/{r['Day']}/S{r['Shift']}: product changed but Prep_min does not equal input Preparation_min ({d.prep[m,p]:.6f})"
                })
            last=p; first_prod=False
    return bad


def validate(d: Data, plan: Plan, unmet_cap: float | None = None) -> list[dict]:
    """Recompute constraints from raw assignments. Never trust solver flags."""
    bad=[]
    def fail(code, message): bad.append({'code':code,'message':message})
    if plan.product.shape!=(d.M,d.T) or plan.quantity.shape!=(d.M,d.T):
        return [{'code':'SHAPE','message':'Plan dimensions do not match input.'}]
    if not np.isfinite(plan.quantity).all(): return [{'code':'NUMBER','message':'Non-finite quantities.'}]
    amounts=np.zeros(d.P); dayqty=defaultdict(float); changes=defaultdict(int)
    runs=defaultdict(list); assigned=np.zeros((d.M,d.P),dtype=int)
    for m in range(d.M):
        last=int(d.initial[m]); seen={last} if last>=0 else set(); first_prod=True
        for t,(day,shift) in enumerate(d.slots):
            p=int(plan.product[m,t]); q=float(plan.quantity[m,t]); where=f'{d.machines[m]["Machine"]}/Day-{day}/S{shift}'
            if p<0:
                if p != -1: fail('PRODUCT_ID',where)
                if abs(q)>TOL: fail('EMPTY_QTY',where)
                continue
            if p>=d.P or q<POSITIVE_T-TOL:
                fail('ACTIVE_QTY',where); continue
            if d.hours[m,t]<=0: fail('CALENDAR',where)
            if d.rate[m,p]<=0: fail('ELIGIBILITY',where); continue
            new_run=p!=last or first_prod
            setup=expected_preparation_min(d,m,p,last,first_prod)
            used=q/d.rate[m,p]+setup/60.
            if used>d.hours[m,t]+TOL: fail('SHIFT_TIME',f'{where}: {used:.8f} > {d.hours[m,t]} h')
            if p!=last:
                if p in seen: fail('NO_RETURN',where)
                for previous in seen:
                    if d.predecessors[p,previous]: fail('SEQUENCE',f'{where}: {p} must precede {previous}')
                if last>=0: changes[m,day]+=1
            seen.add(p); last=p; first_prod=False
            amounts[p]+=q; dayqty[m,p,day]+=q; assigned[m,p]=1
            runs[m,p].append((t,q))
    for (m,p,day),q in dayqty.items():
        if q>d.daily[m,p]+TOL: fail('DAILY_TON',f'{d.machines[m]["Machine"]}/{d.products[p]["Product_ID"]}/Day-{day}: {q:.6f}>{d.daily[m,p]}')
    for (m,day),n in changes.items():
        if n>d.max_changes[m]: fail('DAILY_CHANGEOVER',f'{d.machines[m]["Machine"]}/Day-{day}: {n}>{d.max_changes[m]}')
    for p in range(d.P):
        if amounts[p]>d.target[p]+TOL: fail('SURPLUS',d.products[p]['Product_ID'])
        if d.hard[p] and amounts[p]<d.target[p]-TOL: fail('HARD_TARGET',d.products[p]['Product_ID'])
        if assigned[:,p].sum()>d.product_limits[p]: fail('ALLOCATION_PRODUCT',d.products[p]['Product_ID'])
    for f in range(len(d.families)):
        if np.any(assigned[:,d.family_index==f],axis=1).sum()>d.family_limits[f]: fail('ALLOCATION_FAMILY',d.families[f])
    for (m,p),rows in runs.items():
        q=sum(v for _,v in rows)
        credit=d.initial_qty[m] if d.initial[m]==p else 0.
        if q+credit+TOL>=d.minimum[m,p]: continue
        # All other same-product runs must have ended before this final run starts.
        other=[items for (other_m,other_p),items in runs.items() if other_p==p and other_m!=m]
        temporal=all(max(t for t,_ in items)<rows[0][0] for items in other)
        if not (d.settings['AllowFinalRunBelowMinimum'] and abs(amounts[p]-d.target[p])<=TOL and temporal):
            fail('MIN_RUN',f'{d.machines[m]["Machine"]}/{d.products[p]["Product_ID"]}: {q:.6f}+{credit:.6f} < {d.minimum[m,p]}; not an authorized final remainder')
    met=evaluate(d,plan)
    if unmet_cap is not None and met['unmet']>unmet_cap+TOL: fail('UNMET_CAP',f'{met["unmet"]} > {unmet_cap}')
    return bad


@njit(cache=False)
def _construct_batch(count, seed, rate, minimum, daily, prep, hours, target,
                     pre, init, init_qty, prepared, maxchg, days, fam, priority,
                     allow_final, critical, singlelimits, familylimits):
    """Randomized block construction; no product/machine names are hardcoded."""
    np.random.seed(seed)
    M,P=rate.shape; T=hours.shape[1]
    X=np.full((count,M,T),-1,dtype=np.int16); Q=np.zeros((count,M,T))
    for n in range(count):
        rem=target.copy(); lastend=np.full(P,-1,dtype=np.int64)
        used=np.zeros((M,P),dtype=np.int64)
        morder=np.argsort(np.random.random(M))
        if critical and np.random.random()<0.7:
            # Restricted machines first, random ties.
            keys=np.random.random(M)
            for m in range(M):
                keys[m]+=np.sum(rate[m,:]>0)
            morder=np.argsort(keys)
        for oi in range(M):
            m=morder[oi]; cursor=0; last=init[m]; first=True
            done=np.zeros(P,dtype=np.int64)
            day_changes=np.zeros(np.max(days)+1,dtype=np.int64)
            while cursor<T:
                porder=np.argsort(np.random.random(P))
                chosen=False
                for pi in range(P):
                    p=porder[pi]
                    if rem[p]<POSITIVE_T or rate[m,p]<=0 or done[p]: continue
                    if p==init[m] and not first and last!=p: continue
                    blocked=False
                    for r in range(P):
                        if pre[p,r] and (done[r] or init[m]==r): blocked=True
                    if blocked: continue
                    if np.sum(used[:,p])>=singlelimits[p]: continue
                    f=fam[p]; fammachines=0; already=False
                    for mm in range(M):
                        has=False
                        for pp in range(P):
                            if fam[pp]==f and used[mm,pp]: has=True
                        if has: fammachines+=1
                        if mm==m: already=has
                    if not already and fammachines>=familylimits[f]: continue
                    credit=init_qty[m] if first and init[m]==p else 0.
                    need=max(0.,minimum[m,p]-credit)
                    st=cursor
                    # Target-limited last remainder must follow all other runs of this product.
                    if rem[p]+TOL<need:
                        if not allow_final: continue
                        st=max(st,lastend[p]+1)
                    setup=0. if first and init[m]==p and prepared[m] else prep[m,p]
                    ischange=(last>=0 and p!=last)
                    while st<T:
                        if hours[m,st]>setup/60.+1e-10 and (not ischange or day_changes[days[st]]<maxchg[m]): break
                        st+=1
                    if st>=T: continue
                    goal=rem[p]
                    # Diversify block sizes without weakening minimum-run rules.
                    if goal>need+POSITIVE_T and need>0 and np.random.random()<0.23:
                        goal=need+(goal-need)*np.random.random()
                    temp=np.zeros(T); dy=np.zeros(np.max(days)+1); total=0.; end=-1
                    for t in range(st,T):
                        avail=hours[m,t]-(setup/60. if t==st else 0.)
                        if avail<=0: continue
                        qt=min(avail*rate[m,p],daily[m,p]-dy[days[t]],goal-total)
                        if qt<POSITIVE_T: continue
                        temp[t]=qt; total+=qt; dy[days[t]]+=qt; end=t
                        if total>=goal-TOL: break
                    if total<POSITIVE_T: continue
                    if total+TOL<need:
                        if not (allow_final and abs(total-rem[p])<=TOL and st>lastend[p]): continue
                    for t in range(st,end+1):
                        if temp[t]>0: X[n,m,t]=p; Q[n,m,t]=temp[t]
                    rem[p]=max(0.,rem[p]-total)
                    lastend[p]=max(lastend[p],end)
                    if ischange: day_changes[days[st]]+=1
                    used[m,p]=1; done[p]=1; last=p; first=False; cursor=end+1
                    chosen=True
                    break
                if not chosen: break
    return X,Q


def constructive(d: Data, attempts: int, seed: int, events: list, pool_limit: int=300) -> list[Plan]:
    days=np.array([v[0] for v in d.slots],dtype=np.int64)
    args=(d.rate,d.minimum,d.daily,d.prep,d.hours,d.target,d.predecessors,d.initial,
          d.initial_qty,d.prepared,d.max_changes,days,d.family_index,d.priority,
          d.settings['AllowFinalRunBelowMinimum'],d.settings['CriticalProductFirst'],
          d.product_limits,d.family_limits)
    if NUMBA_AVAILABLE:
        start=time.perf_counter(); _construct_batch(1,seed,*args)
        events.append({'stage':'JIT compile / warm-up (not search)','runtime_sec':time.perf_counter()-start,'status':'COMPLETED','actual_attempts':0})
    start=time.perf_counter(); archive=[]; baseline=float('inf'); valid=0
    completed=0; batchsize=256
    while completed<attempts:
        n=min(batchsize,attempts-completed)
        X,Q=_construct_batch(n,(seed+completed+17)%2147483647,*args)
        # Vectorized early tonnage filter; rich checks only for contenders.
        produced=Q.sum(axis=(1,2)); unmet=d.target.sum()-produced
        for i in range(n):
            if unmet[i]>baseline+d.settings['UnmetTolerance_t']+TOL: continue
            plan=Plan(X[i].astype(np.int64),Q[i].copy(),'CONSTRUCTIVE')
            if validate(d,plan): continue
            valid+=1; met=plan.metrics; key=preference_key(d,met)
            if met['unmet']<baseline:
                baseline=met['unmet']
                archive=[p for p in archive if p.metrics['unmet']<=baseline+d.settings['UnmetTolerance_t']+TOL]
            # Pareto archive: lower unmet and lexicographically better preferences.
            if any(p.metrics['unmet']<=met['unmet']+1e-8 and preference_key(d,p.metrics)<=key for p in archive): continue
            archive=[p for p in archive if not (met['unmet']<=p.metrics['unmet']+1e-8 and key<=preference_key(d,p.metrics))]
            archive.append(plan)
            if len(archive)>pool_limit:
                # Heuristic archive cap is explicitly reported, never an optimality claim.
                archive.sort(key=lambda p:p.metrics['unmet']); keep=[archive[0]]
                keep+=sorted(archive[1:],key=lambda p:preference_key(d,p.metrics))[:pool_limit-1]
                archive=keep
        completed+=n
        if completed==attempts or completed%max(batchsize,(attempts//5//batchsize)*batchsize)==0:
            LOG.info('Heuristic %s/%s attempts; best unmet %.6f t',completed,attempts,baseline)
    events.append({'stage':'Constructive heuristic','runtime_sec':time.perf_counter()-start,
        'status':'COMPLETED','requested_attempts':attempts,'actual_attempts':completed,
        'notes':f'Independent-validated contenders={valid}; Pareto archive={len(archive)}; duplicates possible; archive cap={pool_limit}'})
    return archive


class LinearModel:
    """Small sparse MILP builder with readable constraint coefficients."""
    def __init__(self):
        self.lo=[]; self.hi=[]; self.integer=[]; self.names=[]
        self.rows=[]; self.lower=[]; self.upper=[]
    def var(self,name,lo=0.,hi=1.,integer=False):
        i=len(self.lo); self.lo.append(lo); self.hi.append(hi); self.integer.append(int(integer)); self.names.append(name); return i
    def add(self,coeff,lo=-np.inf,hi=np.inf):
        row={}
        for i,v in coeff.items() if isinstance(coeff,dict) else coeff:
            row[i]=row.get(i,0.)+v
        self.rows.append({i:v for i,v in row.items() if v}); self.lower.append(lo); self.upper.append(hi)
    def array(self,obj):
        v=np.zeros(len(self.lo))
        for i,k in obj.items(): v[i]=k
        return v
    def solve(self,obj,seconds,extra,disp=False):
        rows=self.rows+[r[0] for r in extra]
        lo=self.lower+[r[1] for r in extra]; hi=self.upper+[r[2] for r in extra]
        ii=[]; jj=[]; vv=[]
        for r,row in enumerate(rows):
            for c,v in row.items(): ii.append(r); jj.append(c); vv.append(v)
        A=coo_matrix((vv,(ii,jj)),shape=(len(rows),len(self.lo))).tocsc()
        return milp(self.array(obj),integrality=np.asarray(self.integer),
            bounds=Bounds(self.lo,self.hi),constraints=LinearConstraint(A,lo,hi),
            options={'time_limit':max(.01,seconds),'mip_rel_gap':0.,'presolve':True,'disp':disp})


@dataclass
class ModelPack:
    model: LinearModel
    q: dict
    x: dict
    y: dict
    st: dict
    objectives: dict
    unmet: dict
    first: dict
    last: dict


def make_model(d: Data) -> ModelPack:
    """Unrestricted slot MILP. Machine assignments and block order are free.

    Pairwise disjunctive run intervals enforce no-return across idle slots.
    Quantities are continuous tons; binary assignment activation is >=0.1 kg.
    """
    b=LinearModel(); q={}; x={}; y={}; st={}; first={}; last={}; exception={}; chg={}; total={}; amount={}
    T=d.T; big=T+1
    pairs=[(m,p) for m in range(d.M) for p in range(d.P) if d.rate[m,p]>0 and d.target[p]>=POSITIVE_T]
    mp_for_p={p:[(m,p) for m in range(d.M) if (m,p) in pairs] for p in range(d.P)}
    used_m={}
    for m,p in pairs:
        y[m,p]=b.var(f'y_{m}_{p}',integer=True)
        first[m,p]=b.var(f'first_{m}_{p}',hi=T-1,integer=True)
        last[m,p]=b.var(f'last_{m}_{p}',hi=T-1,integer=True)
        exception[m,p]=b.var(f'final_{m}_{p}',hi=d.settings['AllowFinalRunBelowMinimum'],integer=True)
        amount[m,p]=b.var(f'amount_{m}_{p}',hi=d.target[p])
        starts=[]; xs=[]; qs=[]; earlier=[]
        for t in range(T):
            if d.hours[m,t]<=0: continue
            k=(m,p,t)
            q[k]=b.var(f'q_{m}_{p}_{t}',hi=min(d.target[p],d.rate[m,p]*d.hours[m,t],d.daily[m,p]))
            x[k]=b.var(f'x_{m}_{p}_{t}',integer=True)
            st[k]=b.var(f'start_{m}_{p}_{t}',integer=True)
            setup=0. if d.initial[m]==p and d.prepared[m] else d.prep[m,p]
            b.add({q[k]:1,x[k]:-min(d.target[p],d.rate[m,p]*d.hours[m,t],d.daily[m,p])},hi=0)
            b.add({q[k]:1,x[k]:-POSITIVE_T},lo=0)
            b.add({q[k]:1/d.rate[m,p],st[k]:setup/60.,x[k]:-d.hours[m,t]},hi=0)
            b.add({st[k]:1,x[k]:-1},hi=0)
            b.add([(a,1) for a in earlier]+[(st[k],T)],hi=T)
            b.add({first[m,p]:1,x[k]:T},hi=t+T)
            b.add({last[m,p]:1,x[k]:-T},lo=t-T)
            starts.append(st[k]); xs.append(x[k]); qs.append(q[k]); earlier.append(x[k])
        b.add([(i,1) for i in starts]+[(y[m,p],-1)],lo=0,hi=0)
        b.add([(i,1) for i in xs]+[(y[m,p],-T)],hi=0)
        b.add([(st[m,p,t],t) for t in range(T) if (m,p,t) in st]+[(first[m,p],-1)],lo=0,hi=0)
        b.add({last[m,p]:1,y[m,p]:-(T-1)},hi=0)
        b.add([(i,1) for i in qs]+[(amount[m,p],-1)],lo=0,hi=0)
        b.add({exception[m,p]:1,y[m,p]:-1},hi=0)
        carry=d.initial_qty[m] if d.initial[m]==p else 0.
        needed=max(0.,d.minimum[m,p]-carry)
        b.add({amount[m,p]:1,y[m,p]:-needed,exception[m,p]:needed},lo=0)
        if needed<=POSITIVE_T:
            b.add({exception[m,p]:1},hi=0)
        else:
            # e=1 implies a genuinely below-minimum run, not a gratuitous exception.
            b.add({amount[m,p]:1,exception[m,p]:d.target[p]},hi=needed-POSITIVE_T+d.target[p])
        for day in sorted(set(a for a,_ in d.slots)):
            b.add({q[m,p,t]:1 for t,(dd,_) in enumerate(d.slots) if dd==day and (m,p,t) in q},hi=d.daily[m,p])
    unmet={}
    for p in range(d.P):
        unmet[p]=b.var(f'unmet_{p}',hi=0. if d.hard[p] else d.target[p])
        b.add([(amount[k],1) for k in mp_for_p[p]]+[(unmet[p],1)],lo=d.target[p],hi=d.target[p])
        b.add({exception[k]:1 for k in mp_for_p[p]},hi=1)
        for k in mp_for_p[p]:
            b.add({unmet[p]:1,exception[k]:d.target[p]},hi=d.target[p])
            for other in mp_for_p[p]:
                if other==k: continue
                # Every other run of p must finish in an earlier slot than the final remainder.
                b.add({last[other]:1,first[k]:-1,exception[k]:big,y[other]:big},hi=2*big-1)
    for m in range(d.M):
        mps=[(mm,p) for mm,p in pairs if mm==m]
        used_m[m]=b.var(f'used_machine_{m}',integer=True)
        for k in mps: b.add({y[k]:1,used_m[m]:-1},hi=0)
        b.add([(y[k],1) for k in mps]+[(used_m[m],-1)],lo=0)
        # Initial empty-machine setup does not count as a changeover.
        initial_starts=[]; earlier=[]
        for t in range(T):
            xt=[x[m,p,t] for _,p in mps if (m,p,t) in x]
            stt=[st[m,p,t] for _,p in mps if (m,p,t) in st]
            b.add({i:1 for i in xt},hi=1)
            chg[m,t]=b.var(f'change_{m}_{t}',integer=True)
            coeff=[(i,1) for i in stt]+[(chg[m,t],-1)]
            if d.initial[m]<0:
                a=b.var(f'initial_setup_{m}_{t}',integer=True)
                initial_starts.append(a)
                b.add([(a,1)]+[(i,-1) for i in stt],hi=0)
                b.add([(i,1) for i in earlier]+[(a,T)],hi=T)
                coeff.append((a,-1))
            elif (m,int(d.initial[m]),t) in st:
                coeff.append((st[m,int(d.initial[m]),t],-1))
            b.add(coeff,lo=0,hi=0)
            earlier.extend(xt)
        if d.initial[m]<0:
            b.add([(i,1) for i in initial_starts]+[(used_m[m],-1)],lo=0,hi=0)
        for day in sorted(set(a for a,_ in d.slots)):
            b.add({chg[m,t]:1 for t,(dd,_) in enumerate(d.slots) if day==dd},hi=d.max_changes[m])
        for i,k in enumerate(mps):
            for other in mps[i+1:]:
                # Exactly one ordering when both products use this machine.
                o=b.var(f'order_{m}_{k[1]}_{other[1]}',integer=True)
                b.add({last[k]:1,first[other]:-1,y[k]:big,y[other]:big,o:big},hi=3*big-1)
                b.add({last[other]:1,first[k]:-1,y[k]:big,y[other]:big,o:-big},hi=2*big-1)
        for a in range(d.P):
            for p in range(d.P):
                if d.predecessors[a,p] and (m,a) in y and (m,p) in y:
                    b.add({last[m,a]:1,first[m,p]:-1,y[m,a]:big,y[m,p]:big},hi=2*big-1)
        if d.initial[m]>=0:
            p0=int(d.initial[m])
            for _,p in mps:
                if d.predecessors[p,p0]: b.add({y[m,p]:1},hi=0)
                if p!=p0 and (m,p0) in y:
                    b.add({last[m,p0]:1,first[m,p]:-1,y[m,p0]:big,y[m,p]:big},hi=2*big-1)
    product_used={}; family_used={}; fm={}
    for p in range(d.P):
        product_used[p]=b.var(f'produced_product_{p}',integer=True)
        ys=[y[k] for k in mp_for_p[p]]
        for v in ys: b.add({v:1,product_used[p]:-1},hi=0)
        b.add([(v,1) for v in ys]+[(product_used[p],-1)],lo=0)
        b.add({v:1 for v in ys},hi=int(d.product_limits[p]))
    for f in range(len(d.families)):
        family_used[f]=b.var(f'produced_family_{f}',integer=True)
        for m in range(d.M):
            fm[m,f]=b.var(f'family_machine_{m}_{f}',integer=True)
            ys=[y[m,p] for p in range(d.P) if d.family_index[p]==f and (m,p) in y]
            for v in ys: b.add({v:1,fm[m,f]:-1},hi=0)
            b.add([(v,1) for v in ys]+[(fm[m,f],-1)],lo=0)
            b.add({fm[m,f]:1,family_used[f]:-1},hi=0)
        b.add([(fm[m,f],1) for m in range(d.M)]+[(family_used[f],-1)],lo=0)
        b.add({fm[m,f]:1 for m in range(d.M)},hi=int(d.family_limits[f]))
    product_split={v:1 for v in y.values()}; product_split.update({v:-1 for v in product_used.values()})
    family_split={v:1 for v in fm.values()}; family_split.update({v:-1 for v in family_used.values()})
    objectives={
        'MIN_UNMET':{v:1 for v in unmet.values()},
        'MIN_PRODUCT_MACHINE_SPLITS':product_split,'MIN_CHANGEOVERS':{v:1 for v in chg.values()},
        'MIN_FAMILY_MACHINE_SPLITS':family_split,'MIN_PRODUCTS_PER_MACHINE':{v:1 for v in y.values()},
        'MIN_SURPLUS':{},
        'MIN_MACHINE_PRIORITY_COST':{amount[m,p]:d.priority[m,p]-1 for m,p in pairs},
        'TIE_MASTER_PRIORITY':{amount[m,p]:d.machines[m]['Priority']-1 for m,p in pairs},
    }
    return ModelPack(b,q,x,y,st,objectives,unmet,first,last)


def decode(d: Data, pack: ModelPack, values: np.ndarray, method: str) -> Plan:
    X=np.full((d.M,d.T),-1,dtype=np.int64); Q=np.zeros((d.M,d.T))
    for k,ix in pack.x.items():
        if values[ix]>.5:
            m,p,t=k
            if X[m,t]>=0: raise RuntimeError('MILP decoder: multiple products in a shift.')
            X[m,t]=p; Q[m,t]=max(0.,float(values[pack.q[k]]))
    return Plan(X,Q,method)


def milp_hybrid(d: Data, archive: list[Plan], budget: float, events: list, solver_log: bool=False) -> tuple[Plan,Plan,dict]:
    """Incumbent-preserving hybrid, not a claimed native MIP warm start.

    scipy.optimize.milp has no public x0. Known validated plans are retained
    externally and used as objective cutoffs. Reallocation is unrestricted.
    """
    if not archive:
        empty=Plan(np.full((d.M,d.T),-1,dtype=int),np.zeros((d.M,d.T)),'EMPTY_VALID_PLAN')
        if not validate(d,empty): archive=[empty]
    t0=time.perf_counter(); pack=make_model(d)
    events.append({'stage':'MILP model build','runtime_sec':time.perf_counter()-t0,'status':'COMPLETED',
                   'notes':f'{len(pack.model.lo)} variables; {len(pack.model.rows)} rows; unrestricted assignment/order'})
    meta={'native_mip_start':False,'solver':'SciPy / HiGHS','baseline_optimal':False,
          'preference_optimal':False,'milp_solve_budget_sec':budget,'stages':[]}
    spent=0.; baseline=min(archive,key=lambda p:p.metrics['unmet']) if archive else None
    lookup={'MIN_UNMET':'unmet','MIN_PRODUCT_MACHINE_SPLITS':'product_splits','MIN_CHANGEOVERS':'changeovers',
        'MIN_FAMILY_MACHINE_SPLITS':'family_splits','MIN_PRODUCTS_PER_MACHINE':'assignments','MIN_SURPLUS':'surplus',
        'MIN_MACHINE_PRIORITY_COST':'priority_cost','TIE_MASTER_PRIORITY':'master_priority_cost'}
    def call(oid,seconds,extra):
        nonlocal spent
        if seconds<.02: return None,None
        started=time.perf_counter()
        res=pack.model.solve(pack.objectives[oid],seconds,extra,solver_log)
        elapsed=time.perf_counter()-started; spent+=elapsed
        entry={'stage':'MILP '+oid,'runtime_sec':elapsed,'status':str(res.message),
               'objective':oid,'solver_status_code':int(res.status),
               'objective_value':None if getattr(res,'fun',None) is None else float(res.fun),
               'best_bound':None if getattr(res,'mip_dual_bound',None) is None else float(res.mip_dual_bound),
               'gap':None if getattr(res,'mip_gap',None) is None else float(res.mip_gap),
               'requested_sec':seconds}
        plan=None
        if getattr(res,'x',None) is not None:
            candidate=decode(d,pack,res.x,'MILP_'+oid)
            problems=validate(d,candidate)
            if not problems: plan=candidate
            else: entry['rejected_by_validator']=problems; LOG.warning('MILP candidate rejected: %s',problems[:2])
        entry['independent_validation']='PASS' if plan is not None else 'NO_VALID_CANDIDATE'
        events.append(entry); meta['stages'].append(entry)
        LOG.info('%s: %.3f s; %s; unmet=%s',oid,elapsed,res.message,plan.metrics['unmet'] if plan else 'none')
        return plan,res
    # Allocate most time to throughput, remainder to lexicographic preferences.
    extra=[]
    if baseline is not None: extra=[(pack.objectives['MIN_UNMET'],-np.inf,baseline.metrics['unmet']+1e-7)]
    candidate,res=call('MIN_UNMET',budget*.60,extra)
    if candidate is not None:
        archive.append(candidate)
        if baseline is None or candidate.metrics['unmet']<baseline.metrics['unmet']-1e-7: baseline=candidate
    if res is not None and res.status==0 and candidate is not None:
        meta['baseline_optimal']=True
    if baseline is None:
        if res is not None and res.status==2: raise InputError('No feasible schedule: MILP proved the HARD model infeasible.')
        raise InputError('No feasible schedule found within the budget. This is not proof of infeasibility. Increase MILP time or review HARD targets.')
    baseline=baseline.clone(); baseline.method+=' | BASELINE'
    cap=baseline.metrics['unmet']+d.settings['UnmetTolerance_t']; meta['frozen_unmet_cap_t']=cap
    selected=min((p for p in archive if p.metrics['unmet']<=cap+TOL),key=lambda p:preference_key(d,p.metrics)).clone()
    extra=[(pack.objectives['MIN_UNMET'],-np.inf,cap)]
    stages=[o for o in d.objectives if o!='MIN_UNMET']+['MIN_UNMET','TIE_MASTER_PRIORITY']
    all_proved=True
    for i,oid in enumerate(stages):
        # Freeze each achieved higher-priority value before moving to the next.
        if not pack.objectives[oid]:
            events.append({'stage':'MILP '+oid,'runtime_sec':0.,'status':'CONSTANT_OBJECTIVE','notes':'Surplus is forbidden.'})
            continue
        remaining=max(0.,budget-spent)
        slots_left=sum(bool(pack.objectives[o]) for o in stages[i:])
        seconds=remaining/max(1,slots_left)
        cutoff=selected.metrics[lookup[oid]]
        candidate,res=call(oid,seconds,extra+[(pack.objectives[oid],-np.inf,cutoff+1e-7)])
        if candidate is not None and candidate.metrics['unmet']<=cap+TOL:
            # Full lexicographic comparator prevents a numerically degraded incumbent.
            if preference_key(d,candidate.metrics)<preference_key(d,selected.metrics): selected=candidate
        if res is None or res.status!=0: all_proved=False
        value=selected.metrics[lookup[oid]]
        if oid in ['MIN_PRODUCT_MACHINE_SPLITS','MIN_CHANGEOVERS','MIN_FAMILY_MACHINE_SPLITS','MIN_PRODUCTS_PER_MACHINE']:
            value=round(value)
        extra.append((pack.objectives[oid],-np.inf,value+1e-7))
    meta['preference_optimal']=all_proved
    meta['milp_solve_actual_sec']=spent
    meta['status']='OPTIMAL_ALL_STAGES' if all_proved and meta['baseline_optimal'] else 'FEASIBLE_NOT_PROVEN_OPTIMAL'
    if validate(d,selected,cap): raise RuntimeError('Selected plan failed final independent validation.')
    if validate(d,baseline): raise RuntimeError('Baseline failed final independent validation.')
    return baseline,selected,meta


def detail_rows(d: Data, plan: Plan, plan_id: str) -> list[dict]:
    rows=[]; counter=0
    for m in range(d.M):
        last=int(d.initial[m]); lastday=None; first=True; run=''
        for t,(day,shift) in enumerate(d.slots):
            p=int(plan.product[m,t]); q=float(plan.quantity[m,t]); setup=0.; change=0; cross=0
            previous=d.products[last]['Product_ID'] if last>=0 else ''
            if p>=0:
                if first or p!=last:
                    counter+=1; run=f'{plan_id[-1]}{counter:03d}'
                setup=expected_preparation_min(d,m,p,last,first)
                change=int(last>=0 and p!=last)
                cross=int(change and (lastday is None or lastday!=day))
                last=p; lastday=day; first=False
                rate=d.rate[m,p]; prod=q/rate
            else:
                rate=0.; prod=0.
            rows.append({'Plan_ID':plan_id,'Day':f'Day-{day}','Shift':shift,
                'Machine_ID':d.machines[m]['Machine_ID'],'Machine':d.machines[m]['Machine'],
                'Product_ID':d.products[p]['Product_ID'] if p>=0 else '',
                'Product':d.products[p]['Product'] if p>=0 else '',
                'Family':d.products[p]['Family'] if p>=0 else '',
                'Run_ID':run if p>=0 else '', 'Planned_t':q,'Preparation_min':setup,
                'Previous_Product_ID':previous,'Is_Changeover':change,'Cross_Day':cross,
                'Shift_Status':'PRODUCTION' if p>=0 else ('CLOSED' if not d.hours[m,t] else 'IDLE'),
                'Available_h':d.hours[m,t],'Capacity_tph':rate,'Production_h':prod,
                'Occupied_h':prod+setup/60.,'Idle_h':max(0.,d.hours[m,t]-prod-setup/60.),
                'MaxDaily_t':d.daily[m,p] if p>=0 else 0.,'m':m,'p':p,'t':t})
    dq=defaultdict(float); dc=defaultdict(int)
    for r in rows:
        dq[r['m'],r['p'],r['Day']]+=r['Planned_t']; dc[r['m'],r['Day']]+=r['Is_Changeover']
    for r in rows:
        r['Daily_Planned_t']=dq[r['m'],r['p'],r['Day']] if r['p']>=0 else 0.
        r['Daily_Changeovers']=dc[r['m'],r['Day']]
        r['Slot_Check']='PASS'
    prep_bad=validate_detail_preparation(d,rows)
    if prep_bad:
        raise RuntimeError(f'Preparation-minute validation failed before Excel export: {prep_bad}')
    return rows



class ExcelWriter:
    """Write formulas and verified value caches without requiring desktop Excel."""
    def __init__(self):
        self.wb=Workbook(); self.wb.remove(self.wb.active); self.cache={}
        self.wb.calculation=CalcProperties(calcId=191029,fullCalcOnLoad=True,forceFullCalc=True,calcMode='auto')
    def value(self,ws,row,col,value):
        cell=ws.cell(row,col,value)
        if isinstance(value,str) and value.startswith(('=','+','-','@')):
            cell.data_type='s'  # Product names cannot inject formulas.
        return cell
    def formula(self,ws,row,col,formula,value):
        c=ws.cell(row,col,formula); self.cache[(ws.title,c.coordinate)]=value; return c
    def save(self,path):
        path=Path(path); tmp=path.with_suffix('.building.xlsx')
        self.wb.save(tmp)
        ns={'s':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
        with zipfile.ZipFile(tmp) as src, zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as dst:
            byxml={f'xl/worksheets/sheet{i+1}.xml':ws.title for i,ws in enumerate(self.wb)}
            for info in src.infolist():
                body=src.read(info.filename)
                if info.filename in byxml:
                    root=ET.fromstring(body); sheet=byxml[info.filename]
                    for c in root.findall('.//s:c',ns):
                        key=(sheet,c.attrib['r'])
                        if key not in self.cache: continue
                        value=self.cache[key]
                        v=c.find('s:v',ns)
                        if v is None: v=ET.SubElement(c,'{'+ns['s']+'}v')
                        if value is None: value=''
                        if isinstance(value,(str,)):
                            c.set('t','str'); v.text=value
                        elif isinstance(value,(bool,np.bool_)):
                            c.set('t','b'); v.text='1' if value else '0'
                        else:
                            c.attrib.pop('t',None)
                            if not math.isfinite(float(value)): raise RuntimeError('Invalid formula cache value.')
                            v.text=format(float(value),'.16g')
                    body=ET.tostring(root,encoding='utf-8',xml_declaration=True)
                dst.writestr(info,body)
        tmp.unlink(missing_ok=True)


NAVY='17365D'; TEAL='DDEEF0'; LIGHT='F1F5FA'; ORANGE='FFF2CC'
NUMFMT='#,##0.000;[Red]-#,##0.000;0.000'


def initialize_sheet(ws,title,subtitle,ncols):
    ws.sheet_view.showGridLines=False
    ws.merge_cells(start_row=1,start_column=1,end_row=1,end_column=ncols)
    ws.cell(1,1,title).font=Font(name='Calibri',size=16,bold=True,color='FFFFFF')
    ws.cell(1,1).fill=PatternFill('solid',fgColor=NAVY)
    ws.row_dimensions[1].height=30
    ws.merge_cells(start_row=2,start_column=1,end_row=2,end_column=ncols)
    ws.cell(2,1,subtitle).font=Font(name='Calibri',size=10,color='595959')
    ws.cell(2,1).alignment=Alignment(wrap_text=True,vertical='center')
    ws.row_dimensions[2].height=32
    for c in range(1,ncols+1): ws.column_dimensions[get_column_letter(c)].width=17
    ws.freeze_panes='C6'
    ws.sheet_properties.pageSetUpPr.fitToPage=True
    ws.page_setup.orientation='landscape'; ws.page_setup.paperSize=ws.PAPERSIZE_A3
    ws.page_setup.fitToWidth=1; ws.page_setup.fitToHeight=0


def header(ws,row,labels):
    for col,value in enumerate(labels,1):
        c=ws.cell(row,col,value); c.font=Font(name='Calibri',bold=True,color='FFFFFF',size=10)
        c.fill=PatternFill('solid',fgColor=NAVY)
        c.alignment=Alignment(wrap_text=True,vertical='center')
    ws.row_dimensions[row].height=32


def finish_table(ws,start,end,ncols):
    for r in range(start,end+1):
        ws.row_dimensions[r].height=max(ws.row_dimensions[r].height or 0,24)
        for c in range(1,ncols+1):
            z=ws.cell(r,c)
            z.alignment=Alignment(vertical='center',wrap_text=True)
            z.font=Font(name='Calibri',size=10,color='000000' if z.data_type=='f' else '00643C')
            if r%2==0: z.fill=PatternFill('solid',fgColor=LIGHT)
            if isinstance(z.value,(int,float)) or z.data_type=='f': z.number_format=NUMFMT


def export_excel(d: Data, baseline: Plan, selected: Plan, meta: dict, events: list, config: dict, path: Path) -> None:
    """Export only a compact Dashboard sheet. No detail/result/summary sheets are created."""
    wb = Workbook()
    ws = wb.active
    ws.title = "Dashboard"

    # Only the metrics requested for the final output workbook.
    metrics = [
        ("Total_Target_t", selected.metrics["target"]),
        ("Planned_t", selected.metrics["planned"]),
        ("Unmet_t", selected.metrics["unmet"]),
        ("Fulfillment",
         1 - selected.metrics["unmet"] / selected.metrics["target"]
         if selected.metrics["target"] else 1.0),
        ("Runs", selected.metrics["runs"]),
        ("Changeovers", selected.metrics["changeovers"]),
        ("UnmetTolerance_t", d.settings["UnmetTolerance_t"]),
        ("Heuristic_Attempts", config["heuristic_trials"]),
        ("MILP_Budget_sec", config["milp_seconds"]),
    ]

    # Simple two-column dashboard.
    ws["A1"] = "Metric"
    ws["B1"] = "Value"
    header_fill = PatternFill("solid", fgColor=NAVY)
    for c in ws[1]:
        c.font = Font(name="Calibri", bold=True, color="FFFFFF")
        c.fill = header_fill
        c.alignment = Alignment(horizontal="center", vertical="center")

    for r, (label, value) in enumerate(metrics, start=2):
        ws.cell(r, 1, label)
        ws.cell(r, 2, value)
        ws.cell(r, 1).font = Font(name="Calibri", size=11)
        ws.cell(r, 2).font = Font(name="Calibri", size=11)
        ws.cell(r, 1).alignment = Alignment(vertical="center")
        ws.cell(r, 2).alignment = Alignment(horizontal="right", vertical="center")
        ws.row_dimensions[r].height = 22

    # Number formats.
    for r in (2, 3, 4, 8):
        ws.cell(r, 2).number_format = "0.0000000"
    ws.cell(5, 2).number_format = "0%"
    for r in (6, 7, 9, 10):
        ws.cell(r, 2).number_format = "0"

    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 18
    ws.freeze_panes = "A2"
    ws.sheet_view.showGridLines = False

    path = Path(path)
    tmp = path.with_name(path.stem + ".tmp.xlsx")
    wb.save(tmp)
    os.replace(tmp, path)

    # Final verification: workbook must contain exactly one sheet and no Excel errors.
    check_formula = load_workbook(path, data_only=False)
    check_values = load_workbook(path, data_only=True)
    try:
        if check_formula.sheetnames != ["Dashboard"]:
            raise RuntimeError(f"Unexpected output sheets: {check_formula.sheetnames}")
        for row in check_values["Dashboard"].iter_rows():
            for cell in row:
                if cell.data_type == "e":
                    raise RuntimeError(f"Excel error in Dashboard!{cell.coordinate}")
    finally:
        check_formula.close()
        check_values.close()


def parse_args(argv=None):
    ap=argparse.ArgumentParser(
        description='Excel input -> hybrid production planning -> Excel output. All planning parameters are read from the input workbook.'
    )
    ap.add_argument('input', type=Path, help='v6 Excel-only input workbook (.xlsx)')
    return ap.parse_args(argv)


def main(argv=None) -> int:
    args=parse_args(argv); start=time.perf_counter(); events=[]
    try:
        input_path=args.input.expanduser().resolve()
        if input_path.suffix.lower()!='.xlsx':
            raise InputError('Input extension must be .xlsx')
        output=input_path.with_name(input_path.stem + '_output.xlsx')
        if output==input_path:
            raise InputError('Output must not overwrite input.')
        output.parent.mkdir(parents=True,exist_ok=True)
        if output.exists():
            output.unlink()
        logging.basicConfig(level=logging.INFO,format='%(asctime)s | %(message)s',handlers=[logging.StreamHandler(sys.stdout)],force=True)
        d=load_input(input_path)
        events.append({'stage':'Excel input read / strict validation','runtime_sec':time.perf_counter()-start,'status':'PASS'})
        LOG.info('Input PASS: %s products, %s machines, %s slots per machine; target %.3f t',d.P,d.M,d.T,d.target.sum())
        LOG.info('PREP RULE: product start/changeover consumes machine_product.Preparation_min; same-product continuation consumes 0; idle/closed does not reset product state.')
        for note in d.warnings: LOG.warning(note)

        trials=d.settings['HeuristicTrials']
        budget=d.settings['MILPPolishTimeSec']
        seed=d.settings['RandomSeed']
        if trials<1 or budget<0 or not math.isfinite(budget):
            raise InputError('HeuristicTrials must be positive and MILPPolishTimeSec finite/nonnegative.')

        config={
            'heuristic_trials':trials,
            'milp_seconds':budget,
            'seed':seed,
            'parameter_source':'Excel settings sheet only',
            'input_settings_overridden':False,
        }
        archive=constructive(d,trials,seed,events)
        baseline,selected,meta=milp_hybrid(d,archive,budget,events,False)

        s=time.perf_counter()
        for p in [baseline,selected]:
            bad=validate(d,p,meta['frozen_unmet_cap_t'] if p is selected else None)
            if bad:
                raise RuntimeError(f'Final plan invalid: {bad}')
        events.append({'stage':'Final independent validation','runtime_sec':time.perf_counter()-s,'status':'PASS'})

        s=time.perf_counter()
        export_excel(d,baseline,selected,meta,events,config,output)
        events.append({'stage':'Excel export / formula cache verification','runtime_sec':time.perf_counter()-s,'status':'COMPLETED'})
        meta['wall_clock_to_excel_complete_sec']=time.perf_counter()-start

        LOG.info('BASELINE | planned=%.6f unmet=%.6f',baseline.metrics['planned'],baseline.metrics['unmet'])
        LOG.info('SELECTED | planned=%.6f unmet=%.6f splits=%s changes=%s | VALIDATION PASS',
            selected.metrics['planned'],selected.metrics['unmet'],selected.metrics['product_splits'],selected.metrics['changeovers'])
        LOG.info('STATUS %s | runtime-to-Excel %.3f s | %s',meta['status'],meta['wall_clock_to_excel_complete_sec'],output)
        return 0
    except (InputError,PermissionError,FileNotFoundError) as exc:
        print(f'ERROR: {exc}',file=sys.stderr); return 2
    except KeyboardInterrupt:
        print('Interrupted. No new completed result is claimed.',file=sys.stderr); return 130
    except Exception:
        LOG.exception('Execution failed; do not treat an incomplete file as a valid result.')
        return 1



# Notebook calistirma noktasi
exit_code = main([str(INPUT_XLSX)])
if exit_code != 0:
    raise RuntimeError(f"Planlayici hata kodu ile sonlandi: {exit_code}")

OUTPUT_XLSX = INPUT_XLSX.with_name(INPUT_XLSX.stem + "_output.xlsx")
if not OUTPUT_XLSX.exists():
    raise FileNotFoundError(f"Cikis Excel olusmadi: {OUTPUT_XLSX}")
print("Tamamlandi. Cikis Excel:", OUTPUT_XLSX)


In [ ]:
# 4) Sonucu goster / istersen indir
from pathlib import Path
print("Sonuc dosyasi:", OUTPUT_XLSX)
print("Boyut (KB):", round(Path(OUTPUT_XLSX).stat().st_size / 1024, 1))

# GitHub linki ile giris verdiyseniz cikis Colab /content altinda olusur.
# Bilgisayariniza indirmek icin:
# from google.colab import files
# files.download(str(OUTPUT_XLSX))
